In [1]:
"""
Script Name: pipeline.ipynb
Authores: Walter Arredondo, Carlos Daza, Andrés Coral
Fecha: 2026-10-05
Fase ETL: Orquestación
Descripción:
Obtiene y ejecuta todos los notebooks .ipynb de la fase ETL.
Verificando si la estructura de carpetas es correcta.
También se verifica los archivos generados por cada fase.
Entradas: - Notebooks .ipynb de la fase ETL
Salidas: - Archivos CSV geneados por cada notebook de la fase ETL
Notas: - Ejecutar este script una sola vez
"""

'\nScript Name: pipeline.ipynb\nAuthores: Walter Arredondo, Carlos Daza, Andrés Coral\nFecha: 2026-10-05\nFase ETL: Orquestación\nDescripción:\nObtiene y ejecuta todos los notebooks .ipynb de la fase ETL.\nVerificando si la estructura de carpetas es correcta.\nTambién se verifica los archivos generados por cada fase.\nEntradas: - Notebooks .ipynb de la fase ETL\nSalidas: - Archivos CSV geneados por cada notebook de la fase ETL\nNotas: - Ejecutar este script una sola vez\n'

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


# Estructura del proyecto

Construye todas las rutas que debe tener el proyecto, las cuales son:


*   extract
*   transform
*   data

In [3]:
from pathlib import Path
import subprocess
import sys

# Ruta raíz del proyecto
BASE_DIR = Path('/content/drive/My Drive/Colab Notebooks/Master/1.ETL/entregable3')

# Construir las rutas de extract transform y data
EXTRACT_DIR = BASE_DIR / 'extract'
TRANSFORM_DIR = BASE_DIR / 'transform'
RAW_DIR = BASE_DIR / 'data' / 'raw'
TRANSFORMED_DIR = BASE_DIR / 'data' / 'transformed'

# Mostrar las rutas
print('BASE_DIR:', BASE_DIR)
print('Existe BASE_DIR:', BASE_DIR.exists())
print('EXTRACT_DIR:', EXTRACT_DIR)
print('TRANSFORM_DIR:', TRANSFORM_DIR)
print('RAW_DIR:', RAW_DIR)
print('TRANSFORMED_DIR:', TRANSFORMED_DIR)

BASE_DIR: /content/drive/My Drive/Colab Notebooks/Master/1.ETL/entregable3
Existe BASE_DIR: True
EXTRACT_DIR: /content/drive/My Drive/Colab Notebooks/Master/1.ETL/entregable3/extract
TRANSFORM_DIR: /content/drive/My Drive/Colab Notebooks/Master/1.ETL/entregable3/transform
RAW_DIR: /content/drive/My Drive/Colab Notebooks/Master/1.ETL/entregable3/data/raw
TRANSFORMED_DIR: /content/drive/My Drive/Colab Notebooks/Master/1.ETL/entregable3/data/transformed


# Existencia de la estructura del proyecto

Verificar que las rutas del proyecto existan, mostrando los notebooks que se van a ejecutar.

In [4]:
# Verificar que las carpetas existan
if not BASE_DIR.exists():
    raise FileNotFoundError(f'No existe la carpeta del proyecto: {BASE_DIR}. '
        'Verificar la ubicación en Google Drive.')

if not EXTRACT_DIR.exists():
    raise FileNotFoundError(f'No existe la carpeta de extracción: {EXTRACT_DIR}')

if not TRANSFORM_DIR.exists():
    raise FileNotFoundError(f'No existe la carpeta de transformación: {TRANSFORM_DIR}')

# Obtener los notebooks de extracción y transformación
EXTRACT_NOTEBOOKS = sorted(EXTRACT_DIR.glob('extract_*.ipynb'))
TRANSFORM_NOTEBOOKS = sorted(TRANSFORM_DIR.glob('transform_*.ipynb'))

# Verificar que existen notebooks
if not EXTRACT_NOTEBOOKS:
    raise FileNotFoundError(f'No se encontraron notebooks extract_*.ipynb en {EXTRACT_DIR}')

if not TRANSFORM_NOTEBOOKS:
    raise FileNotFoundError(f'No se encontraron notebooks transform_*.ipynb en {TRANSFORM_DIR}')

# Imprimir los nombres de los notebooks
print('Notebooks de EXTRACT encontrados:')
for nb in EXTRACT_NOTEBOOKS:
    print('  -', nb.name)

print('\nNotebooks de TRANSFORM encontrados:')
for nb in TRANSFORM_NOTEBOOKS:
    print('  -', nb.name)

Notebooks de EXTRACT encontrados:
  - extract_WB_SE4ALL_data.ipynb

Notebooks de TRANSFORM encontrados:
  - transform_WB_SE4ALL_data.ipynb


# Función para ejecutar un notebook .ipynb

Función necesaria para ejecutar todos los notebooks encontrados en las carpetas de ETL, sin modificar en lo absoluto el código fuente.

In [5]:
def ejecutar_notebook(notebook_path, nombre_ejecucion):
    """
    Descripción:
    Ejecuta un notebook .ipynb sin modificar el notebook fuente

    Parámetros:
    - notebook_path: ruta donde está el notebook .ipynb
    - nombre_ejecucion: indica la fase que se está ejecutando, el número de archivo y el nombre del archivo,
      por ejemplo: EXTRACT 1/1 - extract_WB_SE4ALL_data

    Salida:
    Indica si la ejecución fue exitosa o si hubo algún error
    """
    print(f'\nINICIANDO: {nombre_ejecucion}')
    print(f'Notebook: {notebook_path}')

    # Comando para ejecutar el notebook sin afectar el código fuente
    comando = [
        sys.executable, '-m', 'jupyter', 'nbconvert',
        '--to', 'notebook',
        '--execute',
        '--ExecutePreprocessor.timeout=-1',
        '--output-dir=/tmp/etl_ejecuciones',
        str(notebook_path)
    ]

    # Pasar el comando y ejecutarlo
    resultado = subprocess.run(comando, capture_output = True, text = True)

    # Imprimir las salidas del notebook y los posibles errores
    if resultado.stdout:
        print(resultado.stdout)

    if resultado.returncode != 0:
        if resultado.stderr:
            print(resultado.stderr)
        raise RuntimeError(f'La ejecución {nombre_ejecucion} terminó con error. '
            'El orquestador se detuvo para evitar continuar con datos incompletos.')

    print(f'{nombre_ejecucion} finalizó correctamente')

# Fase de Extracción

Se procede a ejecutar los notebooks .ipynb de la fase de extracción, mostrando los csv generados

In [6]:
print(f'\nSe ejecutarán {len(EXTRACT_NOTEBOOKS)} notebooks de EXTRACT')

# Recorrer los notebooks de Extract para ejecutarlos
for i, notebook in enumerate(EXTRACT_NOTEBOOKS, start = 1):
    ejecutar_notebook(notebook, f'EXTRACT {i}/{len(EXTRACT_NOTEBOOKS)} - {notebook.stem}')

# Buscar los csv generados por la fase Extract
raw_files = sorted(RAW_DIR.glob('*.csv'))
if not raw_files:
    raise FileNotFoundError(f'Extract terminó, pero no se encontró ningún CSV en {RAW_DIR}')

# Imprimir los nombres de los csv y su tamaño en bytes
print('\nArchivos RAW encontrados:')
for archivo in raw_files:
    print(f'  - {archivo.name} | {archivo.stat().st_size:,} bytes')


Se ejecutarán 1 notebooks de EXTRACT

INICIANDO: EXTRACT 1/1 - extract_WB_SE4ALL_data
Notebook: /content/drive/My Drive/Colab Notebooks/Master/1.ETL/entregable3/extract/extract_WB_SE4ALL_data.ipynb
EXTRACT 1/1 - extract_WB_SE4ALL_data finalizó correctamente

Archivos RAW encontrados:
  - extract_energia_sostenible_WB_SE4ALL.csv | 1,606,169 bytes


# Fase de Transformación

Se procede a ejecutar los notebooks .ipynb de la fase de transformación, mostrando los csv generados

In [7]:
print(f'\nSe ejecutarán {len(TRANSFORM_NOTEBOOKS)} notebooks de TRANSFORM')

# Recorrer los notebooks de Transform para ejecutarlos
for i, notebook in enumerate(TRANSFORM_NOTEBOOKS, start=1):
    ejecutar_notebook(notebook, f'TRANSFORM {i}/{len(TRANSFORM_NOTEBOOKS)} - {notebook.stem}')

# Buscar los csv generados por la fase Transform
transformed_files = sorted(TRANSFORMED_DIR.glob('*.csv'))
if not transformed_files:
    raise FileNotFoundError(f'Transform terminó, pero no se encontró ningún CSV en {TRANSFORMED_DIR}')

# Imprimir los nombres de los csv y su tamaño en bytes
print('\nArchivos transformados encontrados:')
for archivo in transformed_files:
    print(f'  - {archivo.name} | {archivo.stat().st_size:,} bytes')


Se ejecutarán 1 notebooks de TRANSFORM

INICIANDO: TRANSFORM 1/1 - transform_WB_SE4ALL_data
Notebook: /content/drive/My Drive/Colab Notebooks/Master/1.ETL/entregable3/transform/transform_WB_SE4ALL_data.ipynb
TRANSFORM 1/1 - transform_WB_SE4ALL_data finalizó correctamente

Archivos transformados encontrados:
  - HUB_energia.csv | 71,597 bytes
  - transform_energia_sostenible_WB_SE4ALL.csv | 74,034 bytes


# Resumen

Al final se muestra los archivos ejecutados en el ETL y los csv generados en cada fase

In [8]:
print('ETL FINALIZADO CORRECTAMENTE')
print(f'Extract ejecutados: {len(EXTRACT_NOTEBOOKS)}')
print(f'Transform ejecutados: {len(TRANSFORM_NOTEBOOKS)}')
print(f'Archivos RAW: {len(raw_files)}')
print(f'Archivos TRANSFORMED: {len(transformed_files)}')

print('\nRAW:')
for archivo in raw_files:
    print('  ', archivo)

print('\TRANSFORMED:')
for archivo in transformed_files:
    print('  ', archivo)

ETL FINALIZADO CORRECTAMENTE
Extract ejecutados: 1
Transform ejecutados: 1
Archivos RAW: 1
Archivos TRANSFORMED: 2

RAW:
   /content/drive/My Drive/Colab Notebooks/Master/1.ETL/entregable3/data/raw/extract_energia_sostenible_WB_SE4ALL.csv
\TRANSFORMED:
   /content/drive/My Drive/Colab Notebooks/Master/1.ETL/entregable3/data/transformed/HUB_energia.csv
   /content/drive/My Drive/Colab Notebooks/Master/1.ETL/entregable3/data/transformed/transform_energia_sostenible_WB_SE4ALL.csv


<>:11: SyntaxWarning: invalid escape sequence '\T'
<>:11: SyntaxWarning: invalid escape sequence '\T'
/tmp/ipykernel_537/1429201732.py:11: SyntaxWarning: invalid escape sequence '\T'
  print('\TRANSFORMED:')
